# Step 3: Test non-representative sample of hard and soft cases

This notebook builds a curated non-representative test set for refining the annotation instructions. The goal is to create an informative set of `n = 500` with hard cases, positive and negative crime-frame cases, no crime cases and unseen cases.

The workflow is:

1. Load the outputs from Step 2.1 and Step 2.2.
2. Build a curated Step 3 test set with hard cases, `CRIME_FRAME_NEG`, `CRIME_FRAME_POS`, `NO_CRIME_FRAME` cases and unseen cases using keyword search.
3. Export a clean file for human annotation.
4. Import human labels and calculate F1 scores and Cohen's kappa.
5. Optionally test few-shot prompting on the same test set.

## Step 3.0: Imports and file paths

This section defines the file paths used in Step 3. It expects that Step 2 created the following CSV files:

- `results/crime_frame_neg.csv`
- `results/crime_frame_pos.csv`
- `results/unclear_frame.csv`
- `results/hard_cases.csv`
- one or more full Step 2.1 result files named like `annotation_step2_1_*.csv`


In [11]:
from pathlib import Path
import pandas as pd
import numpy as np
import os
import importlib
import krippendorff
from dotenv import load_dotenv
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    cohen_kappa_score,
    precision_score,
    recall_score
)

In [12]:
load_dotenv()
api_key = os.getenv("CAC_API_KEY")

HOST  = "https://maki.uni-mannheim.de/v1"
MODEL = "ministral-3-14b"

print(f"API key loaded: {'YES' if api_key else 'NO - check your .env file'}")
print(f"Host: {HOST}")
print(f"Model: {MODEL}")

API key loaded: YES
Host: https://maki.uni-mannheim.de/v1
Model: ministral-3-14b


In [13]:
#for model validity check 
import annotation_setup
importlib.reload(annotation_setup)

from annotation_setup import (
    VALID_LABELS,
    instructions,
    reminder,
    annotate,
    parse_label,
    parse_explanation
)

print("Shared annotation setup loaded.")

Shared annotation setup loaded.


In [ ]:
# ── Config ───────────────────────────────────────────────────────────────
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

# Input files from Step 2 (can be found on Robin Branch)
CRIME_FRAME_NEG = RESULTS_DIR / "crime_frame_neg.csv"
CRIME_FRAME_POS = RESULTS_DIR / "crime_frame_pos.csv"
UNCLEAR_FRAME = RESULTS_DIR / "unclear_frame.csv"
HARD_CASES = RESULTS_DIR / "hard_cases.csv"

# Step 3 output files
STEP3_TESTSET_WITH_MODEL_LABELS = RESULTS_DIR / "step3_testset_500_internal_with_model_info.csv"
STEP3_TESTSET_FOR_HUMAN = RESULTS_DIR / "step3_testset_500_for_human_annotation.csv"
STEP3_HUMAN_HARDCASES = RESULTS_DIR / "step3_human_hard_cases_for_discussion.csv"
STEP3_HUMAN_COMPLETED = RESULTS_DIR / "step3_testset_500_human_completed.csv"
STEP3_HUMAN_MODEL_COMPARISON = RESULTS_DIR / "step3_human_model_comparison.csv"
STEP3_MODEL_HARDCASES = RESULTS_DIR / "step3_model_hardcases.csv"
STEP3_EVALUATION = RESULTS_DIR / "step3_evaluation_summary.csv"
STEP3_FEWSHOT_RESULTS = RESULTS_DIR / "step3_fewshot_results.csv"

STEP3_HUMAN_HARDCASES_NEW250 = "results/human_hardcases_for_discussion_new250.csv"
STEP3_TESTSET_FOR_HUMAN_NEW250 = "results/step4_new250_human_labelling.csv"
STEP3_HUMAN_HARDCASES_NEW250 = "results/human_hardcases_for_discussion_new250.csv"
STEP3_HUMAN_COMPLETED_NEW250 = "results/human_completed_ground_truth_new250.csv"
STEP3_HUMAN_COMPLETED_WITH_MODEL_NEW250 = "results/human_completed_ground_truth_new250_with_fresh_model.csv"
STEP3_HUMAN_MODEL_COMPARISON_NEW250 = "results/human_model_comparison_new250.csv"
STEP3_MODEL_HARDCASES_NEW250 = "results/model_hardcases_new250.csv"
STEP3_EVALUATION_NEW250 = "results/evaluation_summary_new250.csv"

In [ ]:
# Test set size and composition -> can be changed!
N_TESTSET = 500
TESTSET_SEED = 123

PCT_HARD = 0.30
PCT_CRIME = 0.40
PCT_NO_CRIME = 0.25
PCT_KEYWORD = 0.05

N_HARD = int(N_TESTSET * PCT_HARD)
N_CRIME = int(N_TESTSET * PCT_CRIME)
N_NO_CRIME = int(N_TESTSET * PCT_NO_CRIME)
N_KEYWORD = N_TESTSET - N_HARD - N_CRIME - N_NO_CRIME
# ─────────────────────────────────────────────────────────────────────────

print("Step 3 setup complete.")
print(f"Target testset size: {N_TESTSET}")
print(f"Hard cases: {N_HARD}")
print(f"Crime frame cases: {N_CRIME}")
print(f"No crime cases: {N_NO_CRIME}")
print(f"Keyword unseen cases: {N_KEYWORD}")


Step 3 setup complete.
Target testset size: 500
Hard cases: 150
Crime frame cases: 200
No crime cases: 125
Keyword unseen cases: 25


## Step 3.1: Load the candidate files from Step 2

We load the separate case files from Step 2.

In [14]:
def read_optional_csv(path):
    if path.exists():
        df_loaded = pd.read_csv(path)
        df_loaded["source_file"] = path.name
        print(f"Loaded {path}: {len(df_loaded)} rows")
        return df_loaded
    else:
        print(f"Missing {path}: using empty dataframe")
        return pd.DataFrame()

crime_neg = read_optional_csv(CRIME_FRAME_NEG)
crime_pos = read_optional_csv(CRIME_FRAME_POS)
unclear = read_optional_csv(UNCLEAR_FRAME)
hard_cases = read_optional_csv(HARD_CASES)

print("\nLoaded candidate files:")
print(f"CRIME_FRAME_NEG: {len(crime_neg)}")
print(f"CRIME_FRAME_POS: {len(crime_pos)}")
print(f"UNCLEAR: {len(unclear)}")
print(f"HARD_CASES: {len(hard_cases)}")


Loaded results/crime_frame_neg.csv: 839 rows
Loaded results/crime_frame_pos.csv: 17 rows
Loaded results/unclear_frame.csv: 0 rows
Loaded results/hard_cases.csv: 323 rows

Loaded candidate files:
CRIME_FRAME_NEG: 839
CRIME_FRAME_POS: 17
UNCLEAR: 0
HARD_CASES: 323


## Step 3.2: Load full Step 2.1 outputs for `NO_CRIME_FRAME` cases


In [15]:
step2_1_files = sorted(RESULTS_DIR.glob("annotation_step2_1_*.csv"))

if not step2_1_files:
    raise FileNotFoundError(
        "No full Step 2.1 files found. Expected files like results/annotation_step2_1_*.csv"
    )

step2_1_all = []

for file in step2_1_files:
    temp = pd.read_csv(file)
    temp["source_file"] = file.name
    step2_1_all.append(temp)

step2_1_all = pd.concat(step2_1_all, ignore_index=True)
step2_1_all = step2_1_all.drop_duplicates(subset=["article_id", "par_index"])

no_crime_candidates = step2_1_all[
    step2_1_all["label"] == "NO_CRIME_FRAME"
].copy()

print(f"Loaded {len(step2_1_files)} Step 2.1 files")
print(f"Unique Step 2.1 rows: {len(step2_1_all)}")
print(f"NO_CRIME_FRAME candidates: {len(no_crime_candidates)}")


Loaded 2 Step 2.1 files
Unique Step 2.1 rows: 19925
NO_CRIME_FRAME candidates: 19076


## Step 3.3: Standardize candidates and build the 500 row test set

The test set includes all available hard cases, unclear cases, negative crime-frame cases, and positive crime-frame cases. The remaining rows are filled with `NO_CRIME_FRAME` cases.

If the interesting cases exceed 500 rows a mixed sample is used.


In [16]:
def standardize_cases(df_cases, case_source, model_label_col="label"):
    if df_cases is None or len(df_cases) == 0:
        return pd.DataFrame(columns=[
            "article_id", "par_index", "group", "text_block_en", "model_label", "case_source"
        ])

    df_cases = df_cases.copy()

    if "text_block_en" not in df_cases.columns and "text_block" in df_cases.columns:
        df_cases["text_block_en"] = df_cases["text_block"]

    if model_label_col not in df_cases.columns:
        if "final_label" in df_cases.columns:
            model_label_col = "final_label"
        else:
            df_cases["model_label"] = ""
            model_label_col = "model_label"

    keep_cols = ["article_id", "par_index", "group", "text_block_en"]
    existing_keep_cols = [c for c in keep_cols if c in df_cases.columns]

    out = df_cases[existing_keep_cols].copy()
    out["model_label"] = df_cases[model_label_col].values
    out["case_source"] = case_source

    for extra_col in ["agreement", "step2_1_label", "final_label", "source_file"]:
        if extra_col in df_cases.columns:
            out[extra_col] = df_cases[extra_col].values

    return out


neg_cases = standardize_cases(crime_neg, "crime_frame_neg", model_label_col="label")
pos_cases = standardize_cases(crime_pos, "crime_frame_pos", model_label_col="label")
unclear_cases = standardize_cases(unclear, "unclear", model_label_col="label")
hard_case_rows = standardize_cases(hard_cases, "hard_case", model_label_col="final_label")
no_crime_rows = standardize_cases(no_crime_candidates, "no_crime", model_label_col="label")


# Combine all candidates without changing the original CSVs
candidate_sets = [
    hard_case_rows,
    neg_cases,
    pos_cases,
    unclear_cases,
    no_crime_rows
]

all_candidates = pd.concat(candidate_sets, ignore_index=True)

source_priority = {
    "hard_case": 1,
    "crime_frame_neg": 2,
    "crime_frame_pos": 2,
    "unclear": 3,
    "no_crime": 4
}

all_candidates["source_priority"] = all_candidates["case_source"].map(source_priority)
all_candidates = all_candidates.sort_values("source_priority")

all_candidates_unique = all_candidates.drop_duplicates(
    subset=["article_id", "par_index"],
    keep="first"
).copy()

print("Before deduplication")
print(all_candidates["case_source"].value_counts())

print("\nAfter deduplication")
print(all_candidates_unique["case_source"].value_counts())


# Pools for percentage based sampling
hard_pool = all_candidates_unique[
    all_candidates_unique["case_source"] == "hard_case"
].copy()

crime_pool = all_candidates_unique[
    all_candidates_unique["case_source"].isin(["crime_frame_neg", "crime_frame_pos"])
].copy()

no_crime_pool = all_candidates_unique[
    all_candidates_unique["case_source"] == "no_crime"
].copy()


# Keyword unseen cases
df_full = pd.read_csv("dataset/all_multi_paragraphs_2022_2026.csv")

security_keywords = [
    "diebstahl", "raub", "gewalt", "polizei", "ermittlung",
    "verdächtig", "tatverdächtig", "festnahme", "terror",
    "extremismus", "bedrohung", "gefahr", "sicherheit",
    "illegal", "abschiebung", "grenze", "angriff",
    "anschlag", "mord", "körperverletzung", "kriminalität",
    "straftat", "schutz", "prävention"
]

pattern = "|".join(security_keywords)

keyword_rows = df_full[
    df_full["text_block"].str.lower().str.contains(pattern, na=False)
].copy()

keyword_rows = standardize_cases(keyword_rows, "keyword_unseen", model_label_col="label")


def row_keys(data):
    return data["article_id"].astype(str) + "_" + data["par_index"].astype(str)


def remove_used(data, used_keys):
    if len(data) == 0:
        return data

    return data[~row_keys(data).isin(used_keys)].copy()


def sample_n(data, n, seed):
    if len(data) == 0:
        return data

    return data.sample(n=min(n, len(data)), random_state=seed).copy()


used_keys = set()

hard_sample = sample_n(hard_pool, N_HARD, TESTSET_SEED)
used_keys.update(row_keys(hard_sample))

crime_pool = remove_used(crime_pool, used_keys)
crime_sample = sample_n(crime_pool, N_CRIME, TESTSET_SEED)
used_keys.update(row_keys(crime_sample))

no_crime_pool = remove_used(no_crime_pool, used_keys)
no_crime_sample = sample_n(no_crime_pool, N_NO_CRIME, TESTSET_SEED)
used_keys.update(row_keys(no_crime_sample))

keyword_rows = remove_used(keyword_rows, used_keys)
keyword_sample = sample_n(keyword_rows, N_KEYWORD, TESTSET_SEED)
used_keys.update(row_keys(keyword_sample))


step3_testset = pd.concat(
    [hard_sample, crime_sample, no_crime_sample, keyword_sample],
    ignore_index=True
).drop_duplicates(subset=["article_id", "par_index"]).reset_index(drop=True)

step3_testset["testset_id"] = range(1, len(step3_testset) + 1)

cols = ["testset_id"] + [c for c in step3_testset.columns if c != "testset_id"]
step3_testset = step3_testset[cols]

step3_testset.to_csv(STEP3_TESTSET_WITH_MODEL_LABELS, index=False)

print(f"Step 3 internal test set saved to {STEP3_TESTSET_WITH_MODEL_LABELS}")
print(f"Rows in Step 3 test set: {len(step3_testset)}")

print("\nCase source distribution:")
print(step3_testset["case_source"].value_counts())

print("\nModel label distribution:")
print(step3_testset["model_label"].value_counts())


Before deduplication
case_source
no_crime           19076
crime_frame_neg      839
hard_case            323
crime_frame_pos       17
Name: count, dtype: int64

After deduplication
case_source
no_crime           19042
crime_frame_neg      555
hard_case            323
crime_frame_pos        5
Name: count, dtype: int64
Step 3 internal test set saved to results/step3_testset_500_internal_with_model_info.csv
Rows in Step 3 test set: 500

Case source distribution:
case_source
crime_frame_neg    197
hard_case          150
no_crime           125
keyword_unseen      25
crime_frame_pos      3
Name: count, dtype: int64

Model label distribution:
model_label
CRIME_FRAME_NEG    298
NO_CRIME_FRAME     171
                    25
CRIME_FRAME_POS      6
Name: count, dtype: int64


## Step 3.4: Export file for human annotation

This file is meant for manual annotation. It hides the model labels so that human annotators are not biased by the model output.

Fill the `human_label` column manually with exactly one of:

- `NO_CRIME_FRAME`
- `CRIME_FRAME_NEG`
- `CRIME_FRAME_POS`

Optional: use `human_notes` for short comments about difficult cases.


In [ ]:
human_export = step3_testset[
    ["testset_id", "article_id", "par_index", "group", "text_block_en"]
].copy()

human_export["label_robin"] = ""
human_export["label_marmee"] = ""

human_export["notes_robin"] = ""
human_export["notes_marmee"] = ""

human_export.to_csv(STEP3_TESTSET_FOR_HUMAN, index=False)

print(f"Human annotation file saved to {STEP3_TESTSET_FOR_HUMAN}")
human_export.head()

Human annotation file saved to results/step3_testset_500_for_human_annotation.csv


,testset_id,article_id,par_index,group,text_block_en,label_robin,label_hamid,label_marmee,notes_robin,notes_hamid,notes_marmee
0,1,OSZ-doc7k2ebrm9wrognjx5oq7,6,RUS,"Anno August Jagdfeld, der im Marien-Cottage in...",,,,,,
1,2,achse_added_fdcc00add3c941db5b52f21d5475839ec9...,16,REFUG,Bayern: [Gruppe 1] identifizieren\n\nBayern ha...,,,,,,
2,3,632712a23dd49b919c1a13db,5,REFUG,In Griechenland werden [Gruppe 1] als Handlang...,,,,,,
3,4,632712b03dd49b919c1f6623,6,JESID,Al-Kuraischi sprengte sich selbst in die Luft\...,,,,,,
4,5,Mmnews_added_c4d96bca88e842d7092f16699f62056e9...,3,REFUG,Der SPD-Vorsitzende Lars Klingbeil hält die Ei...,,,,,,


## Step 3.5: Human reliability and model validity checks

In this step we check the human reliability and model validity for the two 250 samples (n =500) and **the testset (n=600) for Step 4**

**Step 3.5a** creates a usable human consensus label. Because not every row was coded by all three (later two) annotators. The rule is: If at least two annotators chose the same valid label, that label becomes the human consensus. If two annotators coded a row and disagree, or if three annotators all chose different labels, the row is flagged as a human hard case and has to be discussed manually.

**Step 3.5b** checks human reliability. This tells us how consistently the human annotators applied the codebook. We report the percentage of rows with agreement among rows coded by at least two people and, where possible, pairwise Cohen's kappa between annotator pairs.

**Step 3.5c** checks model validity. This compares the model label to the human consensus label. Rows without a usable human consensus are excluded from the model evaluation until they have been resolved manually in `final_human_label`.


### Step 3.5a: Create human consensus labels and flag human hard cases

This block reads the completed human annotation file, cleans the annotator labels, checks for typos, and creates a consensus label. A consensus exists when at least two annotators agree. Rows without consensus are saved to a separate file for manual discussion. Meassuring intercoder reliability. 

In [ ]:
# Step 3.5a: Create human consensus labels and check reliability for first 250 batch (n= 500)
human_annotation = pd.read_csv(STEP3_TESTSET_FOR_HUMAN)

annotator_cols = ["label_robin", "label_marmee"]

# Clean labels
for col in annotator_cols:
    human_annotation[col] = human_annotation[col].astype(str).str.strip()
    human_annotation[col] = human_annotation[col].replace({
        "": np.nan,
        "nan": np.nan,
        "None": np.nan
    })

# Check for invalid labels or typos
for col in annotator_cols:
    invalid = human_annotation[
        human_annotation[col].notna() &
        ~human_annotation[col].isin(VALID_LABELS)
    ]

    if len(invalid) > 0:
        print(f"Warning: {col} has invalid labels:")
        display(invalid[["testset_id", col]].head(20))


def get_consensus_label(row):
    robin = row["label_robin"]
    marmee = row["label_marmee"]

    if pd.isna(robin) or pd.isna(marmee):
        return np.nan

    if robin == marmee:
        return robin

    return "NO_HUMAN_CONSENSUS"


def get_human_agreement(row):
    robin = row["label_robin"]
    marmee = row["label_marmee"]

    if pd.isna(robin) or pd.isna(marmee):
        return np.nan

    return 1.0 if robin == marmee else 0.0


def get_n_annotators(row):
    return sum(pd.notna(row[col]) for col in annotator_cols)


human_annotation["n_human_annotators"] = human_annotation.apply(get_n_annotators, axis=1)
human_annotation["human_consensus_label"] = human_annotation.apply(get_consensus_label, axis=1)
human_annotation["human_agreement"] = human_annotation.apply(get_human_agreement, axis=1)

human_hard_cases = human_annotation[
    human_annotation["human_consensus_label"] == "NO_HUMAN_CONSENSUS"
].copy()

if "final_human_label" not in human_hard_cases.columns:
    human_hard_cases["final_human_label"] = ""

# ti not overwrite hardcases but add them if we annotated more cases 
if os.path.exists(STEP3_HUMAN_HARDCASES):
    existing_cases = pd.read_csv(STEP3_HUMAN_HARDCASES)
    
    # Concatenate old and new cases. Placing 'existing_cases' first combined with keep='first'
    # ensures that rows you already manually annotated in the CSV are prioritized and preserved.
    human_hard_cases = pd.concat([existing_cases, human_hard_cases]).drop_duplicates(subset=['testset_id'], keep='first')
    print(f"-> Successfully merged new hard cases with existing annotations from '{STEP3_HUMAN_HARDCASES}'.")
else:
    print(f"-> No existing file found. Creating a brand new tracking template at '{STEP3_HUMAN_HARDCASES}'.")

human_hard_cases.to_csv(STEP3_HUMAN_HARDCASES, index=False)

print("=== Human consensus creation ===")

print("Number of annotators per row:")
print(human_annotation["n_human_annotators"].value_counts(dropna=False).sort_index())

print("\nConsensus label distribution:")
print(human_annotation["human_consensus_label"].value_counts(dropna=False))

double_coded = human_annotation[human_annotation["n_human_annotators"] == 2].copy()

n_total = len(double_coded)
n_agree = double_coded["human_consensus_label"].isin(VALID_LABELS).sum()
n_disagree = (double_coded["human_consensus_label"] == "NO_HUMAN_CONSENSUS").sum()

percent_agreement = n_agree / n_total if n_total > 0 else np.nan
percent_disagreement = n_disagree / n_total if n_total > 0 else np.nan

print("\n=== Human reliability summary ===")
print(f"Double-coded rows: {n_total}")
print(f"Rows with agreement: {n_agree} ({percent_agreement * 100:.1f}%)")
print(f"Rows without agreement: {n_disagree} ({percent_disagreement * 100:.1f}%)")

print("\nHuman agreement distribution among double-coded rows:")
print(double_coded["human_agreement"].describe().round(2))

# Cohen's kappa and Krippendorff's alpha between the two annotators
pair_data = human_annotation[
    human_annotation["label_robin"].isin(VALID_LABELS) &
    human_annotation["label_marmee"].isin(VALID_LABELS)
].copy()

if len(pair_data) > 0:
    pair_accuracy = accuracy_score(pair_data["label_robin"], pair_data["label_marmee"])
    pair_kappa = cohen_kappa_score(
        pair_data["label_robin"],
        pair_data["label_marmee"],
        labels=VALID_LABELS
    )
    
    # Prepare data structure for Krippendorff (a list of lists/matrix where rows=coders, columns=items)
    reliability_data = [
        pair_data["label_robin"].tolist(),
        pair_data["label_marmee"].tolist()
    ]
    pair_alpha = krippendorff.alpha(reliability_data=reliability_data, level_of_measurement="nominal")
else:
    pair_accuracy = np.nan
    pair_kappa = np.nan
    pair_alpha = np.nan

print("\n=== Pairwise human reliability ===")
print(f"Overlapping valid annotations: {len(pair_data)}")
print(f"Percent agreement: {pair_accuracy * 100:.1f}%")
print(f"Cohen's kappa: {pair_kappa:.3f}") # can be excluded bc use of Krippendorffs 
print(f"Krippendorff's alpha (nominal): {pair_alpha:.3f}")

print(f"\nHuman hard cases without consensus: {len(human_hard_cases)}")
print(f"Saved human hard cases to {STEP3_HUMAN_HARDCASES}")

-> Successfully merged new hard cases with existing annotations from 'results/step3_human_hard_cases_for_discussion.csv'.
=== Human consensus creation ===
Number of annotators per row:
n_human_annotators
0    250
2    250
Name: count, dtype: int64

Consensus label distribution:
human_consensus_label
NaN                   250
CRIME_FRAME_NEG       110
NO_CRIME_FRAME         79
NO_HUMAN_CONSENSUS     58
CRIME_FRAME_POS         3
Name: count, dtype: int64

=== Human reliability summary ===
Double-coded rows: 250
Rows with agreement: 192 (76.8%)
Rows without agreement: 58 (23.2%)

Human agreement distribution among double-coded rows:
count    250.00
mean       0.77
std        0.42
min        0.00
25%        1.00
50%        1.00
75%        1.00
max        1.00
Name: human_agreement, dtype: float64

=== Pairwise human reliability ===
Overlapping valid annotations: 250
Percent agreement: 76.8%
Cohen's kappa: 0.559
Krippendorff's alpha (nominal): 0.560

Human hard cases without consensus: 58
S

In [83]:
# ── Step 3.5a: Create human consensus labels and check reliability for next 250 ──────

human_annotation = pd.read_csv("results/step4_new250_human_labelling.csv")

annotator_cols = ["label_robin", "label_marmee"]

# Clean labels
for col in annotator_cols:
    human_annotation[col] = human_annotation[col].astype(str).str.strip()
    human_annotation[col] = human_annotation[col].replace({
        "": np.nan,
        "nan": np.nan,
        "None": np.nan
    })

# Check for invalid labels or typos
for col in annotator_cols:
    invalid = human_annotation[
        human_annotation[col].notna() &
        ~human_annotation[col].isin(VALID_LABELS)
    ]

    if len(invalid) > 0:
        print(f"Warning: {col} has invalid labels:")
        display(invalid[["testset_id", col]].head(20))


def get_consensus_label(row):
    robin = row["label_robin"]
    marmee = row["label_marmee"]

    if pd.isna(robin) or pd.isna(marmee):
        return np.nan

    if robin == marmee:
        return robin

    return "NO_HUMAN_CONSENSUS"


def get_human_agreement(row):
    robin = row["label_robin"]
    marmee = row["label_marmee"]

    if pd.isna(robin) or pd.isna(marmee):
        return np.nan

    return 1.0 if robin == marmee else 0.0


def get_n_annotators(row):
    return sum(pd.notna(row[col]) for col in annotator_cols)


human_annotation["n_human_annotators"] = human_annotation.apply(get_n_annotators, axis=1)
human_annotation["human_consensus_label"] = human_annotation.apply(get_consensus_label, axis=1)
human_annotation["human_agreement"] = human_annotation.apply(get_human_agreement, axis=1)

human_hard_cases = human_annotation[
    human_annotation["human_consensus_label"] == "NO_HUMAN_CONSENSUS"
].copy()

if "final_human_label" not in human_hard_cases.columns:
    human_hard_cases["final_human_label"] = ""

# Do not overwrite existing hard cases, but add new ones if more cases were annotated
if os.path.exists(STEP3_HUMAN_HARDCASES_NEW250):
    existing_cases = pd.read_csv(STEP3_HUMAN_HARDCASES_NEW250)

    # Existing cases come first so already manually resolved rows are preserved
    human_hard_cases = (
        pd.concat([existing_cases, human_hard_cases])
        .drop_duplicates(subset=["testset_id"], keep="first")
    )

    print(
        f"-> Successfully merged new hard cases with existing annotations from "
        f"'{STEP3_HUMAN_HARDCASES_NEW250}'."
    )
else:
    print(
        f"-> No existing file found. Creating a brand new tracking template at "
        f"'{STEP3_HUMAN_HARDCASES_NEW250}'."
    )

human_hard_cases.to_csv(STEP3_HUMAN_HARDCASES_NEW250, index=False)

print("=== Human consensus creation ===")

print("Number of annotators per row:")
print(human_annotation["n_human_annotators"].value_counts(dropna=False).sort_index())

print("\nConsensus label distribution:")
print(human_annotation["human_consensus_label"].value_counts(dropna=False))

double_coded = human_annotation[
    human_annotation["n_human_annotators"] == 2
].copy()

n_total = len(double_coded)
n_agree = double_coded["human_consensus_label"].isin(VALID_LABELS).sum()
n_disagree = (
    double_coded["human_consensus_label"] == "NO_HUMAN_CONSENSUS"
).sum()

percent_agreement = n_agree / n_total if n_total > 0 else np.nan
percent_disagreement = n_disagree / n_total if n_total > 0 else np.nan

print("\n=== Human reliability summary ===")
print(f"Double-coded rows: {n_total}")
print(f"Rows with agreement: {n_agree} ({percent_agreement * 100:.1f}%)")
print(f"Rows without agreement: {n_disagree} ({percent_disagreement * 100:.1f}%)")

print("\nHuman agreement distribution among double-coded rows:")
print(double_coded["human_agreement"].describe().round(2))

# Cohen's kappa and Krippendorff's alpha between the two annotators
pair_data = human_annotation[
    human_annotation["label_robin"].isin(VALID_LABELS) &
    human_annotation["label_marmee"].isin(VALID_LABELS)
].copy()

if len(pair_data) > 0:
    pair_accuracy = accuracy_score(
        pair_data["label_robin"],
        pair_data["label_marmee"]
    )

    pair_kappa = cohen_kappa_score(
        pair_data["label_robin"],
        pair_data["label_marmee"],
        labels=VALID_LABELS
    )

    reliability_data = [
        pair_data["label_robin"].tolist(),
        pair_data["label_marmee"].tolist()
    ]

    pair_alpha = krippendorff.alpha(
        reliability_data=reliability_data,
        level_of_measurement="nominal"
    )
else:
    pair_accuracy = np.nan
    pair_kappa = np.nan
    pair_alpha = np.nan

print("\n=== Pairwise human reliability ===")
print(f"Overlapping valid annotations: {len(pair_data)}")
print(f"Percent agreement: {pair_accuracy * 100:.1f}%")
print(f"Cohen's kappa: {pair_kappa:.3f}")
print(f"Krippendorff's alpha (nominal): {pair_alpha:.3f}")

print(f"\nHuman hard cases without consensus: {len(human_hard_cases)}")
print(f"Saved human hard cases to {STEP3_HUMAN_HARDCASES_NEW250}")

-> Successfully merged new hard cases with existing annotations from 'results/human_hardcases_for_discussion_new250.csv'.
=== Human consensus creation ===
Number of annotators per row:
n_human_annotators
0     50
2    250
Name: count, dtype: int64

Consensus label distribution:
human_consensus_label
NO_CRIME_FRAME        205
NaN                    50
CRIME_FRAME_NEG        31
NO_HUMAN_CONSENSUS     14
Name: count, dtype: int64

=== Human reliability summary ===
Double-coded rows: 250
Rows with agreement: 236 (94.4%)
Rows without agreement: 14 (5.6%)

Human agreement distribution among double-coded rows:
count    250.00
mean       0.94
std        0.23
min        0.00
25%        1.00
50%        1.00
75%        1.00
max        1.00
Name: human_agreement, dtype: float64

=== Pairwise human reliability ===
Overlapping valid annotations: 250
Percent agreement: 94.4%
Cohen's kappa: 0.786
Krippendorff's alpha (nominal): 0.786

Human hard cases without consensus: 14
Saved human hard cases to re

/var/folders/_j/4qb1m8dx3sz3drhj8g_f02qr0000gn/T/ipykernel_97684/1541018160.py:72: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat([existing_cases, human_hard_cases])


In [ ]:
# ──After solving Hardcases: Create final human completed ground truth (first 250 batch)─────────────────

human_annotation = pd.read_csv(STEP3_TESTSET_FOR_HUMAN)
human_hard_cases = pd.read_csv(STEP3_HUMAN_HARDCASES)

annotator_cols = ["label_robin", "label_marmee"]

for df_tmp in [human_annotation, human_hard_cases]:
    for col in annotator_cols:
        if col in df_tmp.columns:
            df_tmp[col] = df_tmp[col].astype(str).str.strip()
            df_tmp[col] = df_tmp[col].replace({
                "": np.nan,
                "nan": np.nan,
                "None": np.nan
            })

human_hard_cases["final_human_label"] = human_hard_cases["final_human_label"].astype(str).str.strip()
human_hard_cases["final_human_label"] = human_hard_cases["final_human_label"].replace({
    "": np.nan,
    "nan": np.nan,
    "None": np.nan
})

human_annotation["human_consensus_label"] = human_annotation.apply(get_consensus_label, axis=1)

resolved_hard_cases = human_hard_cases[
    human_hard_cases["final_human_label"].isin(VALID_LABELS)
][["testset_id", "final_human_label"]].copy()

human_completed = human_annotation.merge(
    resolved_hard_cases,
    on="testset_id",
    how="left"
)

human_completed["final_human_label"] = human_completed["final_human_label"].fillna(
    human_completed["human_consensus_label"]
)

human_completed = human_completed[
    human_completed["final_human_label"].isin(VALID_LABELS)
].copy()

human_completed.to_csv(STEP3_HUMAN_COMPLETED, index=False)

print(f"Saved completed human ground truth to {STEP3_HUMAN_COMPLETED}")
print(f"Rows in completed ground truth: {len(human_completed)}")
print("\nFinal human label distribution:")
print(human_completed["final_human_label"].value_counts())

Saved completed human ground truth to results/step3_testset_500_human_completed.csv
Rows in completed ground truth: 250

Final human label distribution:
final_human_label
CRIME_FRAME_NEG    135
NO_CRIME_FRAME     108
CRIME_FRAME_POS      7
Name: count, dtype: int64


In [89]:
# ── After solving hard cases: Create final human completed ground truth for new 250 ──

human_annotation = pd.read_csv(STEP3_TESTSET_FOR_HUMAN_NEW250)
human_hard_cases = pd.read_csv(STEP3_HUMAN_HARDCASES_NEW250)

human_annotation.columns = human_annotation.columns.str.strip()
human_hard_cases.columns = human_hard_cases.columns.str.strip()

annotator_cols = ["label_robin", "label_marmee"]


def clean_label_series(series):
    return (
        series
        .astype(str)
        .str.strip()
        .replace({
            "": np.nan,
            "nan": np.nan,
            "NaN": np.nan,
            "None": np.nan,
            "NONE": np.nan
        })
    )


def normalize_testset_id(series):
    numeric = pd.to_numeric(series, errors="coerce")
    return numeric.astype("Int64").astype(str).replace("<NA>", np.nan)


# Drop completely empty rows from the annotation file
human_annotation = human_annotation.dropna(how="all").copy()

# Clean testset_id
human_annotation["testset_id"] = normalize_testset_id(human_annotation["testset_id"])
human_hard_cases["testset_id"] = normalize_testset_id(human_hard_cases["testset_id"])

# Drop rows without a real testset_id
human_annotation = human_annotation[human_annotation["testset_id"].notna()].copy()
human_hard_cases = human_hard_cases[human_hard_cases["testset_id"].notna()].copy()

# Clean annotator labels
for df_tmp in [human_annotation, human_hard_cases]:
    for col in annotator_cols:
        if col in df_tmp.columns:
            df_tmp[col] = clean_label_series(df_tmp[col])

# Clean final hard case labels
if "final_human_label" not in human_hard_cases.columns:
    raise ValueError(
        f"'final_human_label' column is missing in {STEP3_HUMAN_HARDCASES_NEW250}"
    )

human_hard_cases["final_human_label"] = clean_label_series(
    human_hard_cases["final_human_label"]
)

# Recreate consensus labels
human_annotation["human_consensus_label"] = human_annotation.apply(
    get_consensus_label,
    axis=1
)

# Keep only solved hard cases
resolved_hard_cases = human_hard_cases[
    human_hard_cases["final_human_label"].isin(VALID_LABELS)
][["testset_id", "final_human_label"]].copy()

resolved_hard_cases = resolved_hard_cases.drop_duplicates(
    subset=["testset_id"],
    keep="last"
)

# Map manually resolved hard case labels into full file
final_label_map = resolved_hard_cases.set_index("testset_id")["final_human_label"]

human_completed = human_annotation.copy()

human_completed["final_human_label_from_hardcases"] = (
    human_completed["testset_id"].map(final_label_map)
)

# Use hardcase final_human_label first, otherwise use consensus
human_completed["final_human_label"] = human_completed[
    "final_human_label_from_hardcases"
].fillna(
    human_completed["human_consensus_label"]
)

# Keep only rows with valid final labels
human_completed = human_completed[
    human_completed["final_human_label"].isin(VALID_LABELS)
].copy()

human_completed = human_completed.drop(
    columns=["final_human_label_from_hardcases"]
)

human_completed.to_csv(STEP3_HUMAN_COMPLETED_NEW250, index=False)

print(f"Saved completed human ground truth to {STEP3_HUMAN_COMPLETED_NEW250}")
print(f"Rows in original new 250 file after dropping empty rows: {len(human_annotation)}")
print(f"Rows with solved hard cases: {len(resolved_hard_cases)}")
print(f"Rows in completed ground truth: {len(human_completed)}")

print("\nFinal human label distribution:")
print(human_completed["final_human_label"].value_counts())

missing_final = len(human_annotation) - len(human_completed)

print(f"\nRows without final valid human label: {missing_final}")

if missing_final > 0:
    unresolved = human_annotation.merge(
        human_completed[["testset_id"]],
        on="testset_id",
        how="left",
        indicator=True
    )

    unresolved = unresolved[
        unresolved["_merge"] == "left_only"
    ].drop(columns=["_merge"])

    print("\nUnresolved rows preview:")
    display(
        unresolved[
            ["testset_id", "label_robin", "label_marmee", "human_consensus_label"]
        ].head(30)
    )

Saved completed human ground truth to results/human_completed_ground_truth_new250.csv
Rows in original new 250 file after dropping empty rows: 250
Rows with solved hard cases: 14
Rows in completed ground truth: 250

Final human label distribution:
final_human_label
NO_CRIME_FRAME     213
CRIME_FRAME_NEG     37
Name: count, dtype: int64

Rows without final valid human label: 0


## 3.5b Rerun Model Labels on Human Completed Cases

This section reruns the model on exactly the same cases that were completed by the human annotators.

The model only receives the paragraph text, the current annotation instruction from `annotation_setup.py`, and the reminder. It does not receive the human labels. This keeps the model annotation blind to the ground truth.

Each time this cell runs, the existing `model_label` column gets overwritten for the human completed cases. The cell also updates the model explanation, raw model output, and model name. This makes sure that the internal model info file always reflects the newest annotation instruction and the currently selected model.

In [76]:
# Step 3.5b: Rerun model and overwrite model columns in internal model info

human_completed = pd.read_csv(STEP3_HUMAN_COMPLETED)
model_info = pd.read_csv(STEP3_TESTSET_WITH_MODEL_LABELS)

human_completed = human_completed[
    human_completed["final_human_label"].isin(VALID_LABELS)
].copy().reset_index(drop=True)

completed_ids = set(human_completed["testset_id"])

eval_input = model_info[
    model_info["testset_id"].isin(completed_ids)
].copy().reset_index(drop=True)

print(f"Human completed rows with valid final label: {len(human_completed)}")
print(f"Rows in internal model info to rerun: {len(eval_input)}")
print(f"Running fresh model labels with model: {MODEL}")

if len(eval_input) != len(human_completed):
    print("Warning: Number of matched rows differs from human_completed.")
    print("Check whether all testset_id values are present in both files.")

required_cols = [
    "testset_id",
    "text_block_en",
    "model_label"
]

missing_cols = [
    col for col in required_cols
    if col not in model_info.columns
]

if missing_cols:
    raise ValueError(f"Missing columns in STEP3_TESTSET_WITH_MODEL_LABELS: {missing_cols}")

for col in ["model_explanation", "model_raw_output", "model"]:
    if col not in model_info.columns:
        model_info[col] = pd.NA

fresh_labels = {}

for i, row in eval_input.iterrows():
    raw = annotate(
        text=str(row["text_block_en"]),
        instructions=instructions,
        reminder=reminder,
        api_key=api_key,
        HOST=HOST,
        MODEL=MODEL,
        temperature=0.0
    )

    fresh_labels[row["testset_id"]] = {
        "model_label": parse_label(raw),
        "model_explanation": parse_explanation(raw),
        "model_raw_output": raw,
        "model": MODEL
    }

    if (i + 1) % 10 == 0 or (i + 1) == len(eval_input):
        print(f"[{i + 1}/{len(eval_input)}] done")

for testset_id, values in fresh_labels.items():
    mask = model_info["testset_id"] == testset_id

    model_info.loc[mask, "model_label"] = values["model_label"]
    model_info.loc[mask, "model_explanation"] = values["model_explanation"]
    model_info.loc[mask, "model_raw_output"] = values["model_raw_output"]
    model_info.loc[mask, "model"] = values["model"]

model_info.to_csv(STEP3_TESTSET_WITH_MODEL_LABELS, index=False)

print(f"\nUpdated model labels in {STEP3_TESTSET_WITH_MODEL_LABELS}")

print("\nUpdated model label distribution for human completed cases:")
print(
    model_info[
        model_info["testset_id"].isin(completed_ids)
    ]["model_label"].value_counts(dropna=False)
)

Human completed rows with valid final label: 250
Rows in internal model info to rerun: 250
Running fresh model labels with model: ministral-3-14b
[10/250] done
[20/250] done
[30/250] done
[40/250] done
[50/250] done
[60/250] done
[70/250] done
[80/250] done
[90/250] done
[100/250] done
[110/250] done
[120/250] done
[130/250] done
[140/250] done
[150/250] done
[160/250] done
[170/250] done
[180/250] done
[190/250] done
[200/250] done
[210/250] done
[220/250] done
[230/250] done
[240/250] done
[250/250] done

Updated model labels in results/step3_testset_500_internal_with_model_info.csv

Updated model label distribution for human completed cases:
model_label
CRIME_FRAME_NEG    145
NO_CRIME_FRAME     104
CRIME_FRAME_POS      1
Name: count, dtype: int64


In [91]:
# Step 3.5b: Rerun model and overwrite model columns for completed new 250

human_completed = pd.read_csv(STEP3_HUMAN_COMPLETED_NEW250)
human_completed.columns = human_completed.columns.str.strip()

def normalize_testset_id(series):
    numeric = pd.to_numeric(series, errors="coerce")
    return numeric.astype("Int64").astype(str).replace("<NA>", np.nan)

human_completed["testset_id"] = normalize_testset_id(human_completed["testset_id"])
human_completed = human_completed[human_completed["testset_id"].notna()].copy()

human_completed = human_completed[
    human_completed["final_human_label"].isin(VALID_LABELS)
].copy().reset_index(drop=True)

print(f"Human completed rows with valid final label: {len(human_completed)}")
print(f"Running fresh model labels with model: {MODEL}")

required_cols = [
    "testset_id",
    "text_block_en",
    "final_human_label"
]

missing_cols = [
    col for col in required_cols
    if col not in human_completed.columns
]

if missing_cols:
    raise ValueError(
        f"Missing columns in {STEP3_HUMAN_COMPLETED_NEW250}: {missing_cols}"
    )

for col in ["model_label", "model_explanation", "model_raw_output", "model"]:
    if col not in human_completed.columns:
        human_completed[col] = pd.NA

fresh_labels = {}

for i, row in human_completed.iterrows():
    raw = annotate(
        text=str(row["text_block_en"]),
        instructions=instructions,
        reminder=reminder,
        api_key=api_key,
        HOST=HOST,
        MODEL=MODEL,
        temperature=0.0
    )

    fresh_labels[row["testset_id"]] = {
        "model_label": parse_label(raw),
        "model_explanation": parse_explanation(raw),
        "model_raw_output": raw,
        "model": MODEL
    }

    if (i + 1) % 10 == 0 or (i + 1) == len(human_completed):
        print(f"[{i + 1}/{len(human_completed)}] done")

for testset_id, values in fresh_labels.items():
    mask = human_completed["testset_id"] == testset_id

    human_completed.loc[mask, "model_label"] = values["model_label"]
    human_completed.loc[mask, "model_explanation"] = values["model_explanation"]
    human_completed.loc[mask, "model_raw_output"] = values["model_raw_output"]
    human_completed.loc[mask, "model"] = values["model"]

human_completed.to_csv(STEP3_HUMAN_COMPLETED_WITH_MODEL_NEW250, index=False)

print(f"\nSaved completed new 250 file with fresh model labels to {STEP3_HUMAN_COMPLETED_WITH_MODEL_NEW250}")

print("\nUpdated model label distribution:")
print(
    human_completed["model_label"].value_counts(dropna=False)
)

print("\nFinal human label distribution:")
print(
    human_completed["final_human_label"].value_counts(dropna=False)
)

Human completed rows with valid final label: 250
Running fresh model labels with model: ministral-3-14b
[10/250] done
[20/250] done
[30/250] done
[40/250] done
[50/250] done
[60/250] done
[70/250] done
[80/250] done
[90/250] done
[100/250] done
[110/250] done
[120/250] done
[130/250] done
[140/250] done
[150/250] done
[160/250] done
[170/250] done
[180/250] done
[190/250] done
[200/250] done
[210/250] done
[220/250] done
[230/250] done
[240/250] done
[250/250] done

Saved completed new 250 file with fresh model labels to results/human_completed_ground_truth_new250_with_fresh_model.csv

Updated model label distribution:
model_label
NO_CRIME_FRAME     164
CRIME_FRAME_NEG     85
CRIME_FRAME_POS      1
Name: count, dtype: int64

Final human label distribution:
final_human_label
NO_CRIME_FRAME     213
CRIME_FRAME_NEG     37
Name: count, dtype: int64


## 3.5c Validate Updated Model Labels Against Human Ground Truth

This section compares the updated model labels against the final human labels.

The human label `final_human_label` is treated as the ground truth. The model label `model_label` comes from the updated internal model info file created in Step 3.5b.

The section creates two output files. The first file contains all human completed cases with both the human label and the model label, including whether they agree or disagree. The second file contains only the disagreement cases and is used as the model hardcase file for error analysis.

The section also calculates accuracy, macro precision, macro recall, macro F1, weighted F1, and Cohen’s kappa, to check model validity. The metric summary is saved to `step3_evaluation_summary.csv`.

In [ ]:
# Step 3.5c: Validity checks against human ground truth (first 250 batch)

human_completed = pd.read_csv(STEP3_HUMAN_COMPLETED)
model_info = pd.read_csv(STEP3_TESTSET_WITH_MODEL_LABELS)

human_completed = human_completed[
    human_completed["final_human_label"].isin(VALID_LABELS)
].copy().reset_index(drop=True)

completed_ids = set(human_completed["testset_id"])

model_info_eval = model_info[
    model_info["testset_id"].isin(completed_ids)
].copy()

comparison_columns = [
    "testset_id",
    "model",
    "model_label",
    "model_explanation",
    "model_raw_output",
    "case_source",
    "agreement",
    "step2_1_label",
    "final_label",
    "source_file",
    "source_priority"
]

available_comparison_columns = [
    col for col in comparison_columns
    if col in model_info_eval.columns
]

comparison = human_completed.merge(
    model_info_eval[available_comparison_columns],
    on="testset_id",
    how="left"
)

comparison["model_valid_label"] = comparison["model_label"].isin(VALID_LABELS)

comparison["human_model_agree"] = (
    comparison["final_human_label"] == comparison["model_label"]
)

comparison["agreement_status"] = np.where(
    comparison["human_model_agree"],
    "AGREE",
    "DISAGREE"
)

comparison.to_csv(STEP3_HUMAN_MODEL_COMPARISON, index=False)

print(f"Saved human model comparison to {STEP3_HUMAN_MODEL_COMPARISON}")

print("\nAgreement counts:")
print(comparison["agreement_status"].value_counts(dropna=False))

model_hardcases = comparison[
    comparison["agreement_status"] == "DISAGREE"
].copy()

model_hardcases.to_csv(STEP3_MODEL_HARDCASES, index=False)

print(f"\nSaved model hardcases to {STEP3_MODEL_HARDCASES}")
print(f"Number of model hardcases: {len(model_hardcases)}")

eval_data = comparison[
    comparison["final_human_label"].isin(VALID_LABELS) &
    comparison["model_label"].isin(VALID_LABELS)
].copy()

print("\nModel validity against human ground truth")
print(f"Rows used for model evaluation: {len(eval_data)}")
print(f"Rows excluded: {len(comparison) - len(eval_data)}")

print("\nHuman label distribution:")
print(eval_data["final_human_label"].value_counts())

print("\nModel label distribution:")
print(eval_data["model_label"].value_counts())

y_true = eval_data["final_human_label"]
y_pred = eval_data["model_label"]

accuracy = accuracy_score(y_true, y_pred)

macro_precision = precision_score(
    y_true,
    y_pred,
    labels=VALID_LABELS,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    y_true,
    y_pred,
    labels=VALID_LABELS,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    y_true,
    y_pred,
    labels=VALID_LABELS,
    average="macro",
    zero_division=0
)

weighted_f1 = f1_score(
    y_true,
    y_pred,
    labels=VALID_LABELS,
    average="weighted",
    zero_division=0
)

kappa = cohen_kappa_score(
    y_true,
    y_pred,
    labels=VALID_LABELS
)

print("\nSummary metrics:")
print(f"Accuracy: {accuracy:.3f}")
print(f"Macro precision: {macro_precision:.3f}")
print(f"Macro recall: {macro_recall:.3f}")
print(f"Macro F1: {macro_f1:.3f}")
print(f"Weighted F1: {weighted_f1:.3f}")
print(f"Cohen's kappa: {kappa:.3f}")

print("\nConfusion matrix:")
print(confusion_matrix(
    y_true,
    y_pred,
    labels=VALID_LABELS
))

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    labels=VALID_LABELS,
    zero_division=0
))

evaluation_summary = pd.DataFrame([{
    "model": MODEL if "MODEL" in globals() else comparison["model"].dropna().iloc[0] if comparison["model"].notna().any() else None,
    "n_total_human_completed": len(comparison),
    "n_eval": len(eval_data),
    "n_excluded": len(comparison) - len(eval_data),
    "n_agree": int(comparison["human_model_agree"].sum()),
    "n_disagree": int((~comparison["human_model_agree"]).sum()),
    "agreement_rate": comparison["human_model_agree"].mean(),
    "accuracy": accuracy,
    "macro_precision": macro_precision,
    "macro_recall": macro_recall,
    "macro_f1": macro_f1,
    "weighted_f1": weighted_f1,
    "cohens_kappa": kappa
}])

evaluation_summary.to_csv(STEP3_EVALUATION, index=False)

print(f"\nSaved evaluation summary to {STEP3_EVALUATION}")

Saved human model comparison to results/step3_human_model_comparison.csv

Agreement counts:
agreement_status
AGREE       164
DISAGREE     86
Name: count, dtype: int64

Saved model hardcases to results/step3_model_hardcases.csv
Number of model hardcases: 86

Model validity against human ground truth
Rows used for model evaluation: 250
Rows excluded: 0

Human label distribution:
final_human_label
CRIME_FRAME_NEG    135
NO_CRIME_FRAME     108
CRIME_FRAME_POS      7
Name: count, dtype: int64

Model label distribution:
model_label
CRIME_FRAME_NEG    145
NO_CRIME_FRAME     104
CRIME_FRAME_POS      1
Name: count, dtype: int64

Summary metrics:
Accuracy: 0.656
Macro precision: 0.767
Macro recall: 0.490
Macro F1: 0.521
Weighted F1: 0.650
Cohen's kappa: 0.321

Confusion matrix:
[[65 43  0]
 [37 98  0]
 [ 2  4  1]]

Classification report:
                 precision    recall  f1-score   support

 NO_CRIME_FRAME       0.62      0.60      0.61       108
CRIME_FRAME_NEG       0.68      0.73      0.7

In [93]:
# Step 3.5c: Validity checks against human ground truth for new 250
comparison = pd.read_csv(STEP3_HUMAN_COMPLETED_WITH_MODEL_NEW250)
comparison.columns = comparison.columns.str.strip()


def clean_label_series(series):
    return (
        series
        .astype(str)
        .str.strip()
        .replace({
            "": np.nan,
            "nan": np.nan,
            "NaN": np.nan,
            "None": np.nan,
            "NONE": np.nan
        })
    )


def normalize_testset_id(series):
    numeric = pd.to_numeric(series, errors="coerce")
    return numeric.astype("Int64").astype(str).replace("<NA>", np.nan)


required_cols = [
    "testset_id",
    "final_human_label",
    "model_label"
]

missing_cols = [
    col for col in required_cols
    if col not in comparison.columns
]

if missing_cols:
    raise ValueError(
        f"Missing columns in {STEP3_HUMAN_COMPLETED_WITH_MODEL_NEW250}: {missing_cols}"
    )

comparison["testset_id"] = normalize_testset_id(comparison["testset_id"])
comparison["final_human_label"] = clean_label_series(comparison["final_human_label"])
comparison["model_label"] = clean_label_series(comparison["model_label"])

comparison = comparison[
    comparison["testset_id"].notna()
].copy().reset_index(drop=True)

comparison = comparison[
    comparison["final_human_label"].isin(VALID_LABELS)
].copy().reset_index(drop=True)

comparison["model_valid_label"] = comparison["model_label"].isin(VALID_LABELS)

comparison["human_model_agree"] = (
    comparison["final_human_label"] == comparison["model_label"]
)

comparison["agreement_status"] = np.where(
    comparison["human_model_agree"],
    "AGREE",
    "DISAGREE"
)

preferred_columns = [
    "testset_id",
    "group",
    "text_block_en",
    "label_robin",
    "label_marmee",
    "human_consensus_label",
    "final_human_label",
    "model",
    "model_label",
    "model_explanation",
    "model_raw_output",
    "model_valid_label",
    "human_model_agree",
    "agreement_status",
    "article_id"
]

available_preferred_columns = [
    col for col in preferred_columns
    if col in comparison.columns
]

remaining_columns = [
    col for col in comparison.columns
    if col not in available_preferred_columns
]

comparison = comparison[
    available_preferred_columns + remaining_columns
].copy()

comparison.to_csv(STEP3_HUMAN_MODEL_COMPARISON_NEW250, index=False)

print(f"Saved human model comparison to {STEP3_HUMAN_MODEL_COMPARISON_NEW250}")

print("\nAgreement counts:")
print(comparison["agreement_status"].value_counts(dropna=False))

model_hardcases = comparison[
    comparison["agreement_status"] == "DISAGREE"
].copy()

model_hardcases.to_csv(STEP3_MODEL_HARDCASES_NEW250, index=False)

print(f"\nSaved model hardcases to {STEP3_MODEL_HARDCASES_NEW250}")
print(f"Number of model hardcases: {len(model_hardcases)}")

eval_data = comparison[
    comparison["final_human_label"].isin(VALID_LABELS) &
    comparison["model_label"].isin(VALID_LABELS)
].copy()

print("\nModel validity against human ground truth")
print(f"Rows used for model evaluation: {len(eval_data)}")
print(f"Rows excluded: {len(comparison) - len(eval_data)}")

print("\nHuman label distribution:")
print(eval_data["final_human_label"].value_counts(dropna=False))

print("\nModel label distribution:")
print(eval_data["model_label"].value_counts(dropna=False))

if len(eval_data) == 0:
    raise ValueError("No valid rows available for evaluation. Check final_human_label and model_label values.")

y_true = eval_data["final_human_label"]
y_pred = eval_data["model_label"]

accuracy = accuracy_score(y_true, y_pred)

macro_precision = precision_score(
    y_true,
    y_pred,
    labels=VALID_LABELS,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    y_true,
    y_pred,
    labels=VALID_LABELS,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    y_true,
    y_pred,
    labels=VALID_LABELS,
    average="macro",
    zero_division=0
)

weighted_f1 = f1_score(
    y_true,
    y_pred,
    labels=VALID_LABELS,
    average="weighted",
    zero_division=0
)

kappa = cohen_kappa_score(
    y_true,
    y_pred,
    labels=VALID_LABELS
)

print("\nSummary metrics:")
print(f"Accuracy: {accuracy:.3f}")
print(f"Macro precision: {macro_precision:.3f}")
print(f"Macro recall: {macro_recall:.3f}")
print(f"Macro F1: {macro_f1:.3f}")
print(f"Weighted F1: {weighted_f1:.3f}")
print(f"Cohen's kappa: {kappa:.3f}")

print("\nConfusion matrix:")
conf_matrix = confusion_matrix(
    y_true,
    y_pred,
    labels=VALID_LABELS
)

conf_matrix_df = pd.DataFrame(
    conf_matrix,
    index=[f"human_{label}" for label in VALID_LABELS],
    columns=[f"model_{label}" for label in VALID_LABELS]
)

display(conf_matrix_df)

print("\nClassification report:")
print(
    classification_report(
        y_true,
        y_pred,
        labels=VALID_LABELS,
        zero_division=0
    )
)

evaluation_summary = pd.DataFrame([{
    "model": MODEL if "MODEL" in globals() else comparison["model"].dropna().iloc[0] if "model" in comparison.columns and comparison["model"].notna().any() else None,
    "n_total_human_completed": len(comparison),
    "n_eval": len(eval_data),
    "n_excluded": len(comparison) - len(eval_data),
    "n_agree": int(comparison["human_model_agree"].sum()),
    "n_disagree": int((~comparison["human_model_agree"]).sum()),
    "agreement_rate": comparison["human_model_agree"].mean(),
    "accuracy": accuracy,
    "macro_precision": macro_precision,
    "macro_recall": macro_recall,
    "macro_f1": macro_f1,
    "weighted_f1": weighted_f1,
    "cohens_kappa": kappa
}])

evaluation_summary.to_csv(STEP3_EVALUATION_NEW250, index=False)

print(f"\nSaved evaluation summary to {STEP3_EVALUATION_NEW250}")

Saved human model comparison to results/human_model_comparison_new250.csv

Agreement counts:
agreement_status
AGREE       201
DISAGREE     49
Name: count, dtype: int64

Saved model hardcases to results/model_hardcases_new250.csv
Number of model hardcases: 49

Model validity against human ground truth
Rows used for model evaluation: 250
Rows excluded: 0

Human label distribution:
final_human_label
NO_CRIME_FRAME     213
CRIME_FRAME_NEG     37
Name: count, dtype: int64

Model label distribution:
model_label
NO_CRIME_FRAME     164
CRIME_FRAME_NEG     85
CRIME_FRAME_POS      1
Name: count, dtype: int64

Summary metrics:
Accuracy: 0.804
Macro precision: 0.478
Macro recall: 0.590
Macro F1: 0.492
Weighted F1: 0.831
Cohen's kappa: 0.498

Confusion matrix:


,model_NO_CRIME_FRAME,model_CRIME_FRAME_NEG,model_CRIME_FRAME_POS
human_NO_CRIME_FRAME,164,48,1
human_CRIME_FRAME_NEG,0,37,0
human_CRIME_FRAME_POS,0,0,0



Classification report:
                 precision    recall  f1-score   support

 NO_CRIME_FRAME       1.00      0.77      0.87       213
CRIME_FRAME_NEG       0.44      1.00      0.61        37
CRIME_FRAME_POS       0.00      0.00      0.00         0

       accuracy                           0.80       250
      macro avg       0.48      0.59      0.49       250
   weighted avg       0.92      0.80      0.83       250


Saved evaluation summary to results/evaluation_summary_new250.csv


## The next Sections will be calculated in further explained in the notebooks to Step 3 
(Ex.:`framing_annotation_step3_CoT + FSE.ipynb`)

## Step 3.6: Inspect model errors

This section helps identify where the instruction needs improvement. These rows are especially useful for revising the codebook or writing few-shot examples.


In [ ]:
if not STEP3_HUMAN_COMPLETED.exists():
    print("Human labels not available yet.")
else:
    errors = eval_data[
        eval_data["human_consensus_label"] != eval_data["model_label"]
    ].copy()

    print(f"Errors: {len(errors)}")

    display(errors[
        [
            "testset_id",
            "case_source",
            "group",
            "human_consensus_label",
            "model_label",
            "text_block_en"
        ]
    ].head(30))


# Step 3.2: Few-shot examples

Few-shot examples should be manually curated. Ideally, do not copy cases from the Step 3 test set into the few-shot prompt, because that would bias the evaluation. Use invented examples or separate training examples.

Good few-shot examples should include:

- at least one clear case per label
- hard boundary cases, especially victim versus perpetrator
- cases where crime is mentioned but not linked to the group
- cases where terrorism or violence is legitimized by a group
- cases where security or prevention is framed positively

The order should be mixed so the model does not learn a simple label sequence.


In [ ]:
FEW_SHOT_EXAMPLES = [
    {
        "text": "Beispieltext hier einfügen.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Keine explizite Verknüpfung der Gruppe mit Kriminalität oder Sicherheit."
    },
    {
        "text": "Beispieltext hier einfügen.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Die Gruppe wird explizit als Quelle von Gefahr, Kriminalität oder Sicherheitsbedrohung dargestellt."
    },
    {
        "text": "Beispieltext hier einfügen.",
        "label": "CRIME_FRAME_POS",
        "explanation": "Der Text rahmt Sicherheit, Prävention oder Schutz im Zusammenhang mit der Gruppe positiv."
    },
]

def format_few_shot_examples(examples):
    formatted = []
    for ex in examples:
        formatted.append(
            "Text:
"
            + ex["text"]
            + "

Label: "
            + ex["label"]
            + "
Explanation: "
            + ex["explanation"]
        )
    return "

---

".join(formatted)

few_shot_block = format_few_shot_examples(FEW_SHOT_EXAMPLES)
print(few_shot_block)


## Step 3.7: Optional few-shot annotation function

This version adds the few-shot examples to the prompt before the paragraph to annotate. It uses the same output format as the earlier annotation function.

Run this only after you have added real few-shot examples above and after the API variables are available from your setup.


In [ ]:
def annotate_few_shot(text, instructions, reminder, few_shot_examples, temperature=0.0):
    few_shot_text = format_few_shot_examples(few_shot_examples)

    prompt = (
        f"{instructions}

"
        f"Hier sind Beispiele für die Annotation:

{few_shot_text}

"
        f"Jetzt annotieren Sie diesen Absatz:

{text}

"
        f"{reminder}

"
        "Respond in this format exactly:
"
        "Label: <NO_CRIME_FRAME or CRIME_FRAME_NEG or CRIME_FRAME_POS>
"
        "Explanation: <one sentence explaining your choice>"
    )

    payload = {
        "model": MODEL,
        "temperature": temperature,
        "messages": [
            {"role": "system", "content": "Sie sind ein Experte für Inhaltsanalyse. Befolgen Sie die Annotationsanweisungen genau und antworten Sie immer im angegebenen Format."},
            {"role": "user", "content": prompt}
        ]
    }

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    for attempt in range(3):
        try:
            response = requests.post(
                f"{HOST}/chat/completions",
                json=payload,
                headers=headers,
                timeout=120
            )
            if response.status_code == 200:
                return response.json()["choices"][0]["message"]["content"].strip()
            else:
                print(f"  [attempt {attempt+1}] Status {response.status_code}")
        except Exception as e:
            print(f"  [attempt {attempt+1}] Error: {e}")
            time.sleep(5)

    return "ERROR"


## Step 3.8: Optional few-shot run on the Step 3 test set

This runs the few-shot prompt on the Step 3 test set and saves the result. It can then be compared against the human labels using the same metrics as above.


In [ ]:
RUN_FEW_SHOT = False

if RUN_FEW_SHOT:
    fewshot_results = []

    for i, row in step3_testset.iterrows():
        raw = annotate_few_shot(
            str(row["text_block_en"]),
            instructions,
            reminder,
            FEW_SHOT_EXAMPLES,
            temperature=0.0
        )
        label = parse_label(raw)
        explanation = parse_explanation(raw)

        fewshot_results.append({
            "testset_id": row["testset_id"],
            "article_id": row["article_id"],
            "par_index": row["par_index"],
            "group": row["group"],
            "text_block_en": row["text_block_en"],
            "fewshot_raw_output": raw,
            "fewshot_label": label,
            "fewshot_explanation": explanation,
        })

        if (i + 1) % 10 == 0:
            print(f"[{i+1}/{len(step3_testset)}] done")

    fewshot_results = pd.DataFrame(fewshot_results)
    fewshot_results.to_csv(STEP3_FEWSHOT_RESULTS, index=False)
    print(f"Few-shot results saved to {STEP3_FEWSHOT_RESULTS}")
else:
    print("RUN_FEW_SHOT is False. Set it to True when you want to run few-shot annotation.")


## Step 3.9: Optional evaluation of few-shot results

After few-shot annotation and human labels are available, this compares the few-shot labels to the human standard labels.


In [ ]:
if not STEP3_FEWSHOT_RESULTS.exists() or not STEP3_HUMAN_COMPLETED.exists():
    print("Few-shot results or human completed file not available yet.")
else:
    fewshot_results = pd.read_csv(STEP3_FEWSHOT_RESULTS)
    human_completed = pd.read_csv(STEP3_HUMAN_COMPLETED)

    few_eval = fewshot_results.merge(
        human_completed[["testset_id", "human_label"]],
        on="testset_id",
        how="left"
    )

    few_eval["human_label"] = few_eval["human_label"].astype(str).str.strip()
    few_eval_ready = few_eval[few_eval["human_label"].isin(VALID_LABELS)].copy()

    y_true = few_eval_ready["human_label"]
    y_pred = few_eval_ready["fewshot_label"]

    print("=== Few-shot Classification Report ===")
    print(classification_report(y_true, y_pred, labels=VALID_LABELS, zero_division=0))

    print("=== Few-shot Cohen's Kappa ===")
    print(cohen_kappa_score(y_true, y_pred, labels=VALID_LABELS))
